# <mark> 딥다이브에서 사용한 예시를 디벨롭하여 과제로 진행

In [ ]:
!pip install -q google-genai chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 113.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 128.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 126.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is th

# 문서 준비

In [ ]:
from google import genai

client = genai.Client(api_key="제미나이 API 키를 넣으세요")                                        #gemini에서 key 를 발급받아서 넣고 돌리기!

def embed_texts(texts):
    """텍스트 리스트를 Gemini 임베딩 벡터로 변환합니다."""
    result = client.models.embed_content(
        model="gemini-embedding-001", contents=texts
    )
    return [e.values for e in result.embeddings]

In [ ]:
import json
from html.parser import HTMLParser

def load_markdown(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        return f.read()

class HTMLTextExtractor(HTMLParser):
    def __init__(self):
        super().__init__()
        self.texts = []
    def handle_data(self, data):
        self.texts.append(data.strip())
    def get_text(self):
        return " ".join(t for t in self.texts if t)

def load_html(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        html = f.read()
    parser = HTMLTextExtractor()
    parser.feed(html)
    return parser.get_text()

def load_json_reviews(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
    texts = []
    for review in data.get("reviews", []):
        texts.append(f"{review.get('title', '')}: {review.get('content', '')}")
    return "\n".join(texts)

In [ ]:
import re, unicodedata

# 추출된 텍스트를 검색에 적합한 형태로 정리하는 전처리 함수입니다.
def preprocess(text):
    """유니코드 정규화 + 불필요 문자 제거 + 연속 공백/줄바꿈 정리"""
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", "", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r" {2,}", " ", text)

    return text.strip()

In [ ]:
import chromadb

files = [
    {"path": "sample_textbook.md", "loader": load_markdown},
    {"path": "sample_homepage.html", "loader": load_html},
    {"path": "sample_reviews.json", "loader": load_json_reviews},
]

In [ ]:
# 로딩 -> 전처리 -> 청킹
# 최종적으로 벡터 DB에 저장할 청크 정보를 모아 둘 리스트입니다.
all_chunks = []

# 파일을 하나씩 순회하며 텍스트를 추출하고 청크를 만듭니다.
for f in files:
    raw = f["loader"](f["path"])

    cleaned = preprocess(raw)

    chunks = [c.strip() for c in cleaned.split("\n\n") if len(c.strip()) > 20]

    for i, chunk in enumerate(chunks):
        all_chunks.append({
            "text": chunk,
            "source": f["path"],
            "id": f"{f['path']}_{i}"
        })

In [ ]:
# 전체 청크 개수를 확인합니다.
print(f"청크 수: {len(all_chunks)}")

청크 수: 29


In [ ]:
# 임베딩 -> ChromaDB 저장
# 로컬 ChromaDB 클라이언트를 생성합니다.
client_db = chromadb.Client()

collection = client_db.get_or_create_collection(
    name="rag_eval", metadata={"hnsw:space": "cosine"}
)

embeddings = embed_texts([c["text"] for c in all_chunks])

collection.add(
    ids=[c["id"] for c in all_chunks],
    documents=[c["text"] for c in all_chunks],
    embeddings=embeddings,
    metadatas=[{"source": c["source"]} for c in all_chunks]
)

In [ ]:
# 실제로 저장된 청크 수를 확인합니다.
print(f"ChromaDB에 {collection.count()}개 청크 저장")

ChromaDB에 29개 청크 저장


# 평가 데이터셋 준비 + RAG 파이프라인 실행

In [ ]:
# === 평가 데이터셋: 질문-정답 쌍 (사람이 작성) ===
eval_questions = [
    {
        "question": "어댑터즈는 무엇인가요?",
        "ground_truth": "어댑터즈는 스타트업코드에서 운영하는 개발 교재 서빙 서비스입니다."
    },
    {
        "question": "5단 분석법이란 무엇인가요?",
        "ground_truth": "5단 분석법은 기술 용어를 일반 명사, 고유 명사, 사용 이유, 사용 방법, 다른 기술과의 비교로 나누어 설명하는 교수법입니다."
    },
    {
        "question": "어댑터즈는 어떤 분야의 교재를 제공하나요?",
        "ground_truth": "인공지능(AI), 데이터 분석, 웹 개발, 인프라 분야의 교재를 제공합니다."
    },
    {
        "question": "어댑터즈 교재는 어떤 수준의 학습자에게 적합한가요?",
        "ground_truth": "프로그래밍 기초 문법(변수, 조건문, 반복문)을 알고 있는 수준부터 시작할 수 있습니다."
    },
    {
        "question": "어댑터즈에 대한 사용자 평가는 어떤가요?",
        "ground_truth": "평균 평점 8.9/10이며, RAG 개념 정리, 면접 준비, 포트폴리오 작성 등에 도움이 된다는 평가가 많습니다."
    },
]

In [ ]:
# 질문을 임베딩한 뒤, ChromaDB에서 관련 문서 청크를 검색하는 함수입니다.
def retrieve(question, n_results=3):
    q_emb = embed_texts([question])[0]

    results = collection.query(query_embeddings=[q_emb], n_results=n_results)

    return results["documents"][0]

In [ ]:
# ChromaDB에서 검색된 문서 청크(contexts)와 사용자의 질문 (question)을 하나로 묶어서 gemini에게 전달하고
# 그에 맞는 한국어 답변을 생성하는 함수입니다.
def generate(contexts, question):
    ctx_text = "\n---\n".join(contexts)
    prompt = f"""다음 문서를 참고하여 질문에 한국어로 답하세요.
문서에 없는 내용은 '제공된 문서에서 해당 정보를 찾을 수 없습니다'라고 답하세요.

[문서]
{ctx_text}

[질문]
{question}"""
    response = client.models.generate_content(model="gemini-2.5-flash", contents=prompt)

    return response.text.strip()

In [ ]:
import time

# 최종 평가용 데이터셋(question, ground_truth, contexts, answer)을 저장할 리스트입니다.
eval_dataset = []

In [ ]:
# 각 질문에 대해 RAG 파이프라인 실행
# 사람이 작성한 질문-정답 쌍을 하나씩 순회합니다.
for item in eval_questions:

    contexts = retrieve(item["question"])                       # 1. Retrieval 실행 : 현재 질문에 대해 관련 문서 청크(contexts)를 검색합니다.

    time.sleep(3)                                               # 잠시 대기

    answer = generate(contexts, item["question"])              # 2. Generation 실행 : 검색된 문맥과 질문을 바탕으로 답변(answer)을 생성합니다.

    time.sleep(3)

    eval_dataset.append({                                      # 3. 평가 데이터셋 1건 완성 : 사람이 준비한 question, ground_truth와 파이프라인이 생성한 contexts, answer를 함께 저장합니다.
        "question": item["question"],
        "ground_truth": item["ground_truth"],
        "contexts": contexts,
        "answer": answer
    })

    # 현재 질문과 생성된 답변의 앞부분을 출력하여 실행 상태를 확인합니다.
    print(f"Q: {item['question']}")
    print(f"A: {answer[:100]}...")
    print()
    print('==========================================================================')
    print()

Q: 어댑터즈는 무엇인가요?
A: 어댑터즈는 다음 분야의 교재를 제공하며, 그 이름은 '여러 가지를 맞춰주는 도구들' 또는 '다양한 것을 맞춰주는 도구 모음'을 의미합니다....


Q: 5단 분석법이란 무엇인가요?
A: 5단 분석법은 어댑터즈의 모든 교재가 작성되는 독자적인 교수법입니다. 이 교수법은 기술 용어를 일반 명사와 고유 명사로 나누어 설명하고, 사용 이유와 방법, 다른 기술과의 비교까지...


Q: 어댑터즈는 어떤 분야의 교재를 제공하나요?
A: 제공된 문서에서 해당 정보를 찾을 수 없습니다....


Q: 어댑터즈 교재는 어떤 수준의 학습자에게 적합한가요?
A: 어댑터즈 교재는 다음과 같은 수준의 학습자들에게 적합합니다:

*   **입문자 및 초급 학습자:** 프로그래밍 기초 지식만 있는 1학년 학생이나 비전공자도 AI 개념을 잡기 좋으...


Q: 어댑터즈에 대한 사용자 평가는 어떤가요?
A: 어댑터즈에 대한 사용자 평가는 전반적으로 매우 긍정적이며, 특히 개념 이해와 면접 준비에 큰 도움이 된다는 평가가 많습니다. 하지만 일부 개선이 필요한 부분도 언급됩니다.

**긍...




# 🤖 클로드 : 4개 지표 평가 + 진단 (규칙 기반 → 딥다이브 카테고리 분류)

In [ ]:
import re

# 정밀 형태소 분석(konlpy) 대신, 끝에 붙은 조사만 떼어내는 '교육용 근사' 정규화입니다.
JOSA = ("으로","에서","에게","까지","부터","라는","은","는","이","가","을","를","에","의","도","와","과","로","만","요")

def tokens(text):
    """문자열 → 핵심 어절 리스트. 구두점 제거 → 공백 분리 → 끝 조사 제거 → 1글자 토큰 버림."""
    words = re.sub(r"[^\w\s]", " ", text).split()
    out = []
    for w in words:
        for j in JOSA:                       # "어댑터즈는" → "어댑터즈"
            if len(w) > len(j) and w.endswith(j):
                w = w[:-len(j)]
                break
        if len(w) > 1:
            out.append(w)
    return out

In [ ]:
def faithfulness(answer, contexts):
    """답변 내용어 중 검색 문맥에 등장하는 비율 = 문맥 근거성.
    (원본은 문장 통째 일치만 봐서 항상 0 → 어절 겹침으로 완화)"""
    ans = tokens(answer)
    if not ans:
        return 1.0
    ctx = set(tokens(" ".join(contexts)))
    grounded = sum(1 for t in ans if t in ctx)
    return grounded / len(ans)

def answer_relevancy(answer, question):
    """질문 내용어가 답변에서 다뤄지는 비율 = 질문-답변 정합성."""
    q = tokens(question)
    if not q:
        return 1.0
    ans = set(tokens(answer))
    return sum(1 for t in q if t in ans) / len(q)

def context_precision(question, contexts):
    """검색 청크 중 질문 내용어를 포함하는 비율 = 노이즈 적을수록 높음.
    (실제 RAGAS는 상위 순위 가중까지 보지만 여기선 단순 비율로 근사)"""
    q = set(tokens(question))
    if not contexts:
        return 0.0
    relevant = sum(1 for c in contexts if q & set(tokens(c)))
    return relevant / len(contexts)

def context_recall(ground_truth, contexts):
    """정답 내용어가 검색 문맥에 들어있는 비율 = 정답 근거 누락 적을수록 높음."""
    gt = tokens(ground_truth)
    if not gt:
        return 1.0
    ctx = set(tokens(" ".join(contexts)))
    return sum(1 for t in gt if t in ctx) / len(gt)

In [ ]:
def diagnose(f, ar, cp, cr, hi=0.5):
    """4지표 → 딥다이브 카테고리. hi = '높다'로 볼 임계값(교육용, 데이터에 맞게 조정)."""
    retrieval_ok = (cp >= hi) and (cr >= hi)      # 검색 축이 멀쩡한가?
    if retrieval_ok:
        # 검색 OK → 문제가 있다면 생성 단계 = 사례 ①
        if f < hi:
            return "사례①-환각형  (검색 OK인데 답이 문맥 밖)"
        if ar < hi:
            return "사례①-핀트이탈형 (검색 OK인데 질문과 어긋난 답)"
        return "정상 (검색·생성 모두 양호)"
    else:
        # 검색 약함 = 사례 ② (이때 Faithfulness는 ⚠️라 판정에 안 씀)
        if ar >= hi:
            return "사례②  (검색 부적절, 답은 그럴듯)"
        return "사례②/복합 (검색 약함 + 답도 부실)"

In [ ]:
metrics = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
results = []

for item in eval_dataset:
    f  = faithfulness(item["answer"], item["contexts"])
    ar = answer_relevancy(item["answer"], item["question"])
    cp = context_precision(item["question"], item["contexts"])
    cr = context_recall(item["ground_truth"], item["contexts"])
    results.append({
        "question": item["question"],
        "faithfulness": f, "answer_relevancy": ar,
        "context_precision": cp, "context_recall": cr,
        "label": diagnose(f, ar, cp, cr),
    })

print("=== RAG 진단 결과 ===\n")
for r in results:
    print(f"질문: {r['question']}")
    print(f"  F={r['faithfulness']:.2f} | AR={r['answer_relevancy']:.2f} "
          f"| CP={r['context_precision']:.2f} | CR={r['context_recall']:.2f}")
    print(f"  → 진단: {r['label']}\n")

avg = {m: sum(r[m] for r in results) / len(results) for m in metrics}
print("--- 평균 ---")
print(f"  F={avg['faithfulness']:.2f} | AR={avg['answer_relevancy']:.2f} "
      f"| CP={avg['context_precision']:.2f} | CR={avg['context_recall']:.2f}")

=== RAG 진단 결과 ===

질문: 어댑터즈는 무엇인가요?
  F=0.87 | AR=0.50 | CP=0.33 | CR=0.29
  → 진단: 사례②  (검색 부적절, 답은 그럴듯)

질문: 5단 분석법이란 무엇인가요?
  F=0.92 | AR=0.33 | CP=0.67 | CR=0.89
  → 진단: 사례①-핀트이탈형 (검색 OK인데 질문과 어긋난 답)

질문: 어댑터즈는 어떤 분야의 교재를 제공하나요?
  F=0.00 | AR=0.00 | CP=1.00 | CR=0.33
  → 진단: 사례②/복합 (검색 약함 + 답도 부실)

질문: 어댑터즈 교재는 어떤 수준의 학습자에게 적합한가요?
  F=0.72 | AR=0.67 | CP=1.00 | CR=0.40
  → 진단: 사례②  (검색 부적절, 답은 그럴듯)

질문: 어댑터즈에 대한 사용자 평가는 어떤가요?
  F=0.69 | AR=0.80 | CP=0.67 | CR=0.36
  → 진단: 사례②  (검색 부적절, 답은 그럴듯)

--- 평균 ---
  F=0.64 | AR=0.46 | CP=0.73 | CR=0.45


In [ ]:
# 진단이 맞는지 실제 답변·문맥을 직접 보고 확인합니다. (지표는 신호일 뿐)
for item, r in zip(eval_dataset, results):
    print(f"질문: {item['question']}  →  {r['label']}")
    print(f"  [정답]   {item['ground_truth'][:120]}")
    print(f"  [답변]   {item['answer'][:120]}")
    print(f"  [문맥 1] {item['contexts'][0][:120]}")
    print()
    print('==========================================================================')
    print()

질문: 어댑터즈는 무엇인가요?  →  사례②  (검색 부적절, 답은 그럴듯)
  [정답]   어댑터즈는 스타트업코드에서 운영하는 개발 교재 서빙 서비스입니다.
  [답변]   어댑터즈는 다음 분야의 교재를 제공하며, 그 이름은 '여러 가지를 맞춰주는 도구들' 또는 '다양한 것을 맞춰주는 도구 모음'을 의미합니다.
  [문맥 1] 어댑터즈는 다음 분야의 교재를 제공합니다.


질문: 5단 분석법이란 무엇인가요?  →  사례①-핀트이탈형 (검색 OK인데 질문과 어긋난 답)
  [정답]   5단 분석법은 기술 용어를 일반 명사, 고유 명사, 사용 이유, 사용 방법, 다른 기술과의 비교로 나누어 설명하는 교수법입니다.
  [답변]   5단 분석법은 어댑터즈의 모든 교재가 작성되는 독자적인 교수법입니다. 이 교수법은 기술 용어를 일반 명사와 고유 명사로 나누어 설명하고, 사용 이유와 방법, 다른 기술과의 비교까지 체계적으로 정리하는 방식입니다.
  [문맥 1] 어댑터즈의 모든 교재는 5단 분석법이라는 독자적인 교수법으로 작성됩니다.
5단 분석법은 기술 용어를 일반 명사와 고유 명사로 나누어 설명하고, 사용 이유와 방법, 다른 기술과의 비교까지 체계적으로 정리하는 방식입니다


질문: 어댑터즈는 어떤 분야의 교재를 제공하나요?  →  사례②/복합 (검색 약함 + 답도 부실)
  [정답]   인공지능(AI), 데이터 분석, 웹 개발, 인프라 분야의 교재를 제공합니다.
  [답변]   제공된 문서에서 해당 정보를 찾을 수 없습니다.
  [문맥 1] 어댑터즈는 다음 분야의 교재를 제공합니다.


질문: 어댑터즈 교재는 어떤 수준의 학습자에게 적합한가요?  →  사례②  (검색 부적절, 답은 그럴듯)
  [정답]   프로그래밍 기초 문법(변수, 조건문, 반복문)을 알고 있는 수준부터 시작할 수 있습니다.
  [답변]   어댑터즈 교재는 다음과 같은 수준의 학습자들에게 적합합니다:

*   **입문자 및 초급 학습자:** 프로그래밍 기초 지식만 있는 1학년 학생이나 비전공자도 A

In [ ]:
# 벡터 DB에 저장된 (문서 텍스트 → 출처 파일) 사전을 만듭니다.  👈 새 변수
# collection.get()은 로컬 DB를 읽는 거라 임베딩/API 호출이 없습니다.
stored = collection.get(include=["documents", "metadatas"])
chunk_to_source = {doc: meta["source"] for doc, meta in zip(stored["documents"], stored["metadatas"])}

for item, r in zip(eval_dataset, results):
    print(f"질문: {item['question']}  →  {r['label']}")
    print(f"  [정답]   {item['ground_truth'][:120]}")
    print(f"  [생성된 답변]   {item['answer'][:120]}")
    print(f"  [검색된 문맥]")
    for i, ctx in enumerate(item["contexts"], 1):                  # 검색된 청크 전부 출처와 함께
        src = chunk_to_source.get(ctx, "출처 불명")
        print(f"    {i}. ({src})  {ctx[:90]}")
    print()
    print('=' * 74)
    print()

질문: 어댑터즈는 무엇인가요?  →  사례②  (검색 부적절, 답은 그럴듯)
  [정답]   어댑터즈는 스타트업코드에서 운영하는 개발 교재 서빙 서비스입니다.
  [답변]   어댑터즈는 다음 분야의 교재를 제공하며, 그 이름은 '여러 가지를 맞춰주는 도구들' 또는 '다양한 것을 맞춰주는 도구 모음'을 의미합니다.
  [검색된 문맥]
    1. (sample_textbook.md)  어댑터즈는 다음 분야의 교재를 제공합니다.
    2. (sample_textbook.md)  Adapter는 '적응시키는 것, 맞춰주는 도구'라는 뜻입니다.
-z는 복수형 또는 브랜드 이름에 사용되는 접미사입니다.
일반 명사 차원에서 Adapterz를 유
    3. (sample_textbook.md)  | 단어 | 내용 |
|---|---|
| Adapter | 적응시키는 것, 맞춰주는 도구 |
| -z | 복수형 또는 브랜드화를 나타내는 접미사 |
| Adap


질문: 5단 분석법이란 무엇인가요?  →  사례①-핀트이탈형 (검색 OK인데 질문과 어긋난 답)
  [정답]   5단 분석법은 기술 용어를 일반 명사, 고유 명사, 사용 이유, 사용 방법, 다른 기술과의 비교로 나누어 설명하는 교수법입니다.
  [답변]   5단 분석법은 어댑터즈의 모든 교재가 작성되는 독자적인 교수법입니다. 이 교수법은 기술 용어를 일반 명사와 고유 명사로 나누어 설명하고, 사용 이유와 방법, 다른 기술과의 비교까지 체계적으로 정리하는 방식입니다.
  [검색된 문맥]
    1. (sample_textbook.md)  어댑터즈의 모든 교재는 5단 분석법이라는 독자적인 교수법으로 작성됩니다.
5단 분석법은 기술 용어를 일반 명사와 고유 명사로 나누어 설명하고, 사용 이유와 방법,
    2. (sample_textbook.md)  | 단계 | 분석 | 설명 |
|---|---|---|
| 1 | 일반 명사 | 사전적 의미를 먼저 파악합니다. |
| 2 | 고유 명사 | 업계에서 사용하는 기
    

# <font color='red'>1. 배운 점</font>

- #### 요약 : 제미나이 API를 발급받고, 자료를 제공함으로써 RAG가 이루어지는 과정과 평가하는 과정까지 한 번에 볼 수 있어서 좋은 경험이었다. 특히, 추가로 찾아본 자료에서 오픈소스 라이브러리로 'RAGAS'라는게 존재하며, 이름 때문에 헷갈릴 수는 있지만 규칙기반과 LLM as a Judge가 있다는 점도 확인하였다.

<font color='green'>

1.   RAGAS 4지표 — 검색 축·생성 축, 그리고 각 지표의 "비교 쌍"
- 4지표는 두 축으로 갈림: **검색 품질**(Context Precision·Recall)과 **생성 품질**(Faithfulness·Answer Relevancy). 진단의 핵심 규칙은 "점수 낮은 지표 = 범인이 있는 단계".
- 각 지표가 *무엇과 무엇을* 비교하는지 정리: Precision(`question↔contexts`), Recall(`ground_truth↔contexts`), Faithfulness(`contexts↔answer`), Answer Relevancy(`question↔answer`).

2.   문제 진단 — 정상 / 환각형 / 핀트이탈형 / 검색 문제
- 검색이 멀쩡(CP·CR↑)한데 답이 틀리면 **생성 문제**: 지어내면 Faithfulness↓(환각형), 질문 핀트를 놓치면 Answer Relevancy↓(핀트이탈형).
- 답이 자연스러워도(AR↑) 검색 지표가 낮으면(CP·CR↓) **검색 문제(사례②)** — 자연스러움에 속으면 안 됨.
- **Faithfulness = 근거성이지 정답성이 아님**: 모델이 나쁜 문맥을 충실히 따르면 F는 높은데 답은 틀릴 수 있어, 항상 Context Recall과 교차 확인해야 함.

3.   "파일에 정답이 있다 ≠ 검색이 가져온다" — 파이프라인 단계 추적
- 원본 파일 → 로더 → 청킹 → 임베딩 → DB → **검색** → 생성. 정답이 파일에 있어도 *로더·청킹 단계*에서 샐 수 있고, 그건 검색(사례②) 이전의 문제임.
- 검색은 **파일 단위가 아니라 청크 단위** — 어댑터즈 정의가 md에 있어도 정의 청크가 top-3 밖으로 밀리면 검색 실패. JSON 로더가 평균 평점 8.9를 안 읽어 인덱스에서 누락된 사례도 확인.

4.   규칙 기반 vs LLM-as-judge
- 우리가 짠 건 RAGAS 지표를 "글자/어절 매칭으로 흉내 낸 **규칙 기반** " 버전이고, **오픈소스 RAGAS 라이브러리는 LLM을 심판으로** 써서 의미·패러프레이즈까지 판단함.
- 둘의 차이는 비교 방법(`in` 연산자 → LLM 심판)일 뿐, **비교 쌍(census)은 동일**하다는 층위를 정리.

</font>

# <font color='red'>2. 어려웠던 점</font>

- #### 요약 : 교재에 올라와있는 코드와 자료를 기반으로 진행하였더니 json에서 title, context만 뽑고, 한국어 조사는 분리하지 않아 제대로 된 RAGAS 를 수행하기 어려웠다. 하지만, 오히려 그런 부분들도 세세하게 확인을 해야 RAG에 대한 제대로된 평가를 진행할 수 있음을 역설적으로 확인할 수 있게 되었다.

<font color='green'>

1.   Faithfulness가 전부 0.00 — "환각"이 아니라 측정의 한계
- `simple_faithfulness`가 답변 문장이 문맥에 *글자 그대로* 들어있는지(`sent in ctx`)만 봐서, 풀어쓴(paraphrase) 답변은 무조건 0. "모든 답이 환각"이 아니라 지표 코드가 깨진 것임을 파악.

2.   한국어 조사로 키워드 매칭이 깨짐
- `question.split()` 토큰이 조사를 달고 있어("어댑터즈는") 문맥의 "어댑터즈가"와 안 맞음 → Answer Relevancy가 거짓으로 0이 나오기도. 조사 제거 근사로 완화.

3.   진단 라벨이 단계를 뭉뚱그림
- 규칙 기반 라벨은 8.9 누락(적재 단계 문제)을 "사례②(검색 부적절)"로 오라벨. *검색이 부적절한 게 아니라 검색 이전(로더)에서 데이터가 샌* 것이라, 라벨을 곧이곧대로 믿으면 오진함을 확인.


</font>

# <font color='red'>3. 앞으로의 보완점 - 🤖 클로드 제시</font>

<font color='green'>

1.   **실제 RAGAS 라이브러리로 돌려보기** — `evaluate(dataset, metrics, evaluator_llm)`로 LLM 심판 점수를 내고, 우리가 손으로 만든 규칙 기반 점수와 비교(특히 Faithfulness 차이).

2.   **노트북 실행으로 사례 검증** — 새로 만든 5개 질문이 의도한 사례(정상/사례②/핀트이탈/환각)로 실제 나오는지 확인. 환각형은 모델이 정직하게 거부하면 안 나올 수 있어, 그 결과도 분석 대상.

3.   **개선 루프 돌리기** — 검색 처방(리랭커·top-k)·생성 처방(그라운딩·질문 의도 명시)을 시스템 차원에서 한 번 적용하고, *같은 평가셋*으로 재측정해 지표가 전반적으로 오르는지 확인.

</font>

# <font color='red'>4. 수행 과정 종합 평가</font>

<font color='green'>

1.   서론
- 본 과제는 **RAGAS 4지표로 RAG 시스템의 문제 원인을 진단**하는 것으로, 특히 "검색은 적절한데 답이 부정확한 경우"와 "답은 자연스러운데 검색이 부적절한 경우"를 각 지표 관점에서 분석했습니다. 어댑터즈 자료(md·html·json)로 검색+생성 파이프라인과 평가를 직접 구현했습니다(Colab).

2.   본론
- 4지표의 비교 쌍(census)을 먼저 정리해 검색 축·생성 축으로 나누고, 이를 정상·환각형·핀트이탈형·사례②로 분류하는 진단 틀을 세웠습니다.
- 코드를 붙여 돌리며 규칙 기반 지표의 한계를 직접 확인했고(Faithfulness 전부 0), "파일에 정답이 있어도 로더·청킹·검색 랭킹 단계에서 샐 수 있다"는 파이프라인 관점으로 원인 위치를 좁혔습니다.
- 마지막으로 실무 RAGAS가 LLM-as-judge라는 점, 즉 비교 쌍은 같고 비교 *방법*만 LLM으로 바뀐다는 차이를 정리했습니다.

3.   결론
- 지표를 외운 것이 아니라 **어느 단계가 범인인지 점수로 읽어내는 진단 능력**을 체득한 것이 가장 값진 수확이었습니다.
- 특히 Faithfulness가 근거성이지 정답성이 아니라는 점, 그리고 정답이 파일에 있어도 검색이 못 가져올 수 있다는 단계 구분이 핵심이었습니다.
- 규칙 기반을 직접 구현해보며 *왜 실무가 LLM 심판을 쓰는지*를 몸으로 이해했고, 남은 라이브러리 실행과 개선 루프는 이 토대 위에서 이어갈 계획입니다.

</font>



🤖 클로드 대화 : 딥다이브 -RAGAS

수행시간 : 약 7시간